# Transformer Foundation：从 PyTorch 到 Attention

这份 Notebook 记录了从 PyTorch 基础逐步理解 Transformer 的过程。
重点是读懂张量形状、模块、梯度更新和手写 Attention，再进入下一份 MiniGPT 实现。

按顺序执行所有单元格即可；示例默认在 CPU 上运行，不需要下载数据。
本文用 `B` 表示 batch size，`T` 表示序列长度，`D` 表示 embedding 维度。

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(42)

## 1. PyTorch 基础

Transformer 将一批 token 表示为 `(B, T, D)` 的张量：每个样本有 `T` 个 token，
每个 token 有 `D` 个特征。`shape` 描述各轴的含义，而矩阵乘法承担线性投影和注意力计算。

下面保留原来的矩阵乘法例子：`(2, 3) @ (3, 4) → (2, 4)`，相邻的内部维度必须相同。
批量矩阵乘法则把同样的规则用于每个样本或每个注意力头。
Transformer 大量使用这些运算，让所有位置的投影与训练时的注意力计算能够并行进行。

In [2]:
A = torch.tensor([[1,2,3], [4,5,6]])
B = torch.tensor([[1,2,3,4], [5,6,7,8], [9,10,11,12]])
C = A @ B
print(C)
print(C.shape)

tensor([[ 38,  44,  50,  56],
        [ 83,  98, 113, 128]])
torch.Size([2, 4])


In [3]:
# 模拟 batch 中的 token 特征；transpose 交换轴，不改变特征数。
hidden_states = torch.randn(2, 3, 4)
transposed = hidden_states.transpose(-2, -1)
scores = hidden_states @ transposed
print("hidden states:", hidden_states.shape)  # (B, T, D)
print("transposed:", transposed.shape)        # (B, D, T)
print("scores:", scores.shape)                # (B, T, T)

hidden states: torch.Size([2, 3, 4])
transposed: torch.Size([2, 4, 3])
scores: torch.Size([2, 3, 3])


## 2. PyTorch 模型结构

`nn.Linear(10, 5)` 对最后一维应用可学习的线性变换：`y = xWᵀ + b`。
在 Attention 中，Q、K、V 就由三个独立的线性层从输入特征投影得到。
对 `(B, T, D)` 输入使用 `nn.Linear(D, D)` 时，batch 和序列轴保持不变。

In [4]:
from torch import nn
layer = nn.Linear(10, 5)
x = torch.randn(2, 10)
y = layer(x)
print(y.shape)

torch.Size([2, 5])


### nn.Module、forward 与 parameter

`nn.Module` 把子层和参数封装成一个模型；调用 `model(x)` 会进入 `forward`。
下面保留原来的 `MyNet`，用两层 Linear 和 ReLU 演示模块组合。
下一份 Notebook 中的 Attention、FFN、DecoderBlock 和 MiniGPT 都沿用这个组织方式。

`named_parameters()` 列出可训练参数，optimizer 更新的正是这些权重与偏置。
注册为 buffer 的位置编码会随模型移动设备，但不会作为参数被 optimizer 更新。

In [5]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.layer2(x)
        return x

model = MyNet()
x = torch.randn(5, 10)
y = model(x)
print(y.shape)

for name, param in model.named_parameters():
    print(name)
    print(param.shape)

torch.Size([5, 1])
layer1.weight
torch.Size([20, 10])
layer1.bias
torch.Size([20])
layer2.weight
torch.Size([1, 20])
layer2.bias
torch.Size([1])


## 3. Training Loop

保留原来的求和回归任务：输入 `(100, 10)`，目标是每行之和 `(100, 1)`。
这里直接复用上一节的 `MyNet`，它与原 `SumNet` 的层结构及 forward 相同。

```text
forward → loss → backward → optimizer.step（更新参数）
```

`zero_grad()` 清除上一步的梯度；`backward()` 根据当前 loss 计算梯度；
`step()` 更新参数。此处用 MSE，MiniGPT 将用交叉熵预测下一个 token。

后续 SFT、LoRA、DPO 也共享这条梯度优化主线，但训练数据、损失与更新参数不同：
SFT 使用监督示例，LoRA 通常冻结原权重并学习低秩增量，DPO 使用偏好对和参考模型构造目标。
本项目只实现基础训练流程，不实现这些扩展方法。

In [6]:
x = torch.randn(100, 10)
y = x.sum(dim=1,keepdim=True)


model = MyNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    y_hat = model(x)
    loss = loss_fn(y_hat, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 12.2681245803833
Epoch 10, Loss: 8.325732231140137
Epoch 20, Loss: 3.082244634628296
Epoch 30, Loss: 0.4689091444015503
Epoch 40, Loss: 0.362192302942276
Epoch 50, Loss: 0.13122476637363434
Epoch 60, Loss: 0.07902205735445023
Epoch 70, Loss: 0.044780805706977844
Epoch 80, Loss: 0.03187726065516472
Epoch 90, Loss: 0.025682246312499046


## 4. Attention Mechanism

注意力可以理解为：每个 Query 根据与 Key 的匹配程度，对 Value 做加权汇总。

- **Q（Query）**：当前位置要查询的特征。
- **K（Key）**：供 Query 匹配的特征。
- **V（Value）**：匹配后实际汇总的信息。

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

除以 `sqrt(d_k)` 控制点积尺度；Softmax 沿 Key 轴计算，使每个 Query 的权重和为 1。
以下保留手写例子：1 个 Query 查询 3 个 Key/Value。
Q/K/V 直接给定，下一份 Notebook 再学习它们的线性投影。

| 步骤 | 输入 shape | 输出 shape |
|---|---|---|
| `K.transpose(1, 2)` | K `(1, 3, 2)` | `(1, 2, 3)` |
| `Q @ Kᵀ / sqrt(d_k)` | Q `(1, 1, 2)`、Kᵀ `(1, 2, 3)` | score `(1, 1, 3)` |
| `softmax(scores, dim=-1)` | `(1, 1, 3)` | weights `(1, 1, 3)` |
| `weights @ V` | weights `(1, 1, 3)`、V `(1, 3, 2)` | output `(1, 1, 2)` |

In [7]:
Q = torch.tensor([[[1., 0.]]])
K = torch.tensor([[
    [1., 0.],
    [0., 1.],
    [1., 1.]
]])                           
V = torch.tensor([[
    [10., 0.],
    [0., 20.],
    [30., 30.]
]])         

d = Q.shape[-1]
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d)
weights = torch.softmax(scores, dim=-1)
output = torch.bmm(weights, V)

print("Scores:", scores)
print("Weights:", weights)
print("Output:", output)
print("权重和:", weights.sum(dim=-1))

Scores: tensor([[[0.7071, 0.0000, 0.7071]]])
Weights: tensor([[[0.4011, 0.1978, 0.4011]]])
Output: tensor([[[16.0445, 15.9889]]])
权重和: tensor([[1.]])


## 5. Embedding

```text
token id → embedding vector → Transformer 输入
```

token id 是词表索引，没有可以直接比较的语义距离；`nn.Embedding` 为每个 id 查找一个
可学习向量。输入必须是整数索引，且在 `[0, vocab_size)` 内。
`(B, T)` 的 token ids 变成 `(B, T, D)` 的特征，随后才能进入线性投影与 Attention。
同一个 id 初始查到同一个向量，经过训练后向量会随模型参数更新。

In [8]:
vocab_size = 1000
embed_dim = 32
embedding = nn.Embedding(vocab_size, embed_dim)
x = torch.tensor([[1, 2, 3]])
output = embedding(x)
print("token ids:", x.shape)
print("embedding vectors:", output.shape)

token ids: torch.Size([1, 3])
embedding vectors: torch.Size([1, 3, 32])


下一步：[MiniGPT From Scratch](02_miniGPT_from_scratch.ipynb)。
在已经理解的 embedding 与 Attention 上加入位置编码、多个头和 causal mask，
再组合成 Decoder、完成 next-token 训练与自回归生成。